# veritract — verified extraction demo

**Verified extraction: truth-anchored LLM structured output.**

LLMs can hallucinate plausible-looking JSON. veritract adds a grounding layer that
traces every extracted value back to a character span in the source document,
and quarantines anything it can't verify — giving you a per-field trust signal
rather than an all-or-nothing response.

---

### How it works

```
(optional) Prompt optimization
        ↓
LLM extraction  ──  GBNF constrained decoding
        ↓
Grounding  ──  fuzzy matching → LLM re-verification
        ↓
Typed provenance per field  +  quarantine
```

**GBNF (GGML Backus-Naur Form)** is the grammar format used by
[llama.cpp](https://github.com/ggerganov/llama.cpp) for constrained token sampling.
When a JSON schema is provided, it is compiled to a GBNF grammar and invalid tokens
are masked to zero probability at every decoding step — so the model physically
*cannot* produce malformed JSON or omit required fields.

**Local inference via [Ollama](https://ollama.com):** the model runs entirely on
your machine. No API key, no data leaving your network. Supports Metal (Mac),
CUDA (Linux/Windows), and CPU.

---

### Input formats

| Format | How |
|---|---|
| Plain text / markdown | `extract(text, schema, llm)` |
| PDF (text + tables + OCR) | `extract_pdf(path, schema, llm)` — via docling |
| Images / figures | `extract(caption, schema, llm, images=[b64])` |
| Mixed (image + text) | combine `images=` with any text source |

---

### Provenance types

| Type | Meaning |
|---|---|
| `direct` | Exact substring — value appears verbatim in source |
| `paraphrased` | Fuzzy match above threshold |
| `inferred` | LLM confirmed; fuzzy matching failed (abbreviation, unit conversion) |
| quarantined | Neither could verify — surfaced for human review |

## Setup

**Running this notebook:**
1. Make sure [Ollama](https://ollama.com) is running: `ollama serve`
2. Pull the model: `ollama pull gemma4:e4b`
3. Select the **veritract (venv)** kernel (Kernel menu → Change Kernel)

The venv has all dependencies installed including `docling` for PDF and figure support.
To install in a fresh environment (not yet on PyPI):
```
git clone https://github.com/LanGuo/veritract.git
cd veritract && pip install -e '.[pdf]'
```

In [ ]:
import textwrap, json
from veritract import extract, extract_raw, ground, LLMClient, MockLLM, load_images_b64

llm = LLMClient(model="gemma4:e4b", temperature=0.0, seed=42)

# Clinical trial abstract used throughout the demo
ABSTRACT = """\
A double-blind randomised controlled trial evaluated the efficacy of atorvastatin
40 mg daily versus placebo in 486 patients with moderate hypercholesterolaemia.
Participants were followed for 52 weeks. The primary endpoint was reduction in
LDL-cholesterol from baseline, which decreased by 43% in the atorvastatin group
compared to 4% in the placebo group (p < 0.0001). Treatment was well tolerated;
elevated liver enzymes were recorded in 3 patients (0.6%).""".strip()

---
## Part 1 — Define your schema

The schema is standard **JSON Schema**. Each property is a field you want extracted.
All `required` fields always appear in the output — GBNF constrained decoding forces
the model to emit every field. If the model is uncertain it returns an empty string,
which surfaces as quarantined rather than silently missing.

```python
schema = {
    "type": "object",
    "properties": {
        "field_name": {"type": "string"},   # add as many fields as you need
        ...
    },
    "required": ["field_name", ...],        # all required fields always emitted
}
```

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "drug":            {"type": "string"},
        "sample_size":     {"type": "string"},
        "primary_outcome": {"type": "string"},
    },
    "required": ["drug", "sample_size", "primary_outcome"],
}

print("Schema:")
print(json.dumps(SCHEMA, indent=2))

---
## Part 2 — Write your prompt and extract

The prompt describes **what each field means** and how to extract it.
Key rules to always include:
- Copy verbatim phrases — no paraphrasing or synthesis
- Return an empty string if a field isn't present (not "N/A" or "not found")
- Don't use prior knowledge — extract only from the provided text

If you omit `prompt`, veritract generates a default one from the schema field names.

In [ ]:
PROMPT = """\
You are extracting facts from a clinical trial abstract.
Rules:
- Copy the exact verbatim phrase from the text. Do not paraphrase or use prior knowledge.
- If a field is not present in this excerpt, return an empty string.

Fields:
* drug: Drug name and dose as stated (e.g. "atorvastatin 40 mg daily").
* sample_size: Number of patients enrolled as stated.
* primary_outcome: Primary endpoint or outcome measure as stated."""

# mode="fuzzy": fuzzy grounding only — fast, no second LLM call.
# mode="full":  also runs LLM re-verification on quarantined fields (higher recall, ~2× slower).
result = extract(ABSTRACT, SCHEMA, llm, prompt=PROMPT, mode="fuzzy")

print("Extracted fields:")
for field, gf in result.extracted.items():
    ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
    print(f"  [{ptype:>12}]  {field}: {gf['value']!r}")

if result.quarantined:
    print("\nQuarantined:")
    for q in result.quarantined:
        print(f"  {q['field_name']}: {q['value']!r} — {q['reason']}")

In [ ]:
# Every grounded field has a precise character span — shown as colored highlight
from IPython.display import HTML, display

_COLORS = {"direct": "#c8f7c5", "paraphrased": "#c5e8f7", "inferred": "#f7f0c5"}

def highlight_html(text, field_name, result):
    gf = result.extracted.get(field_name)
    if not gf or not gf["span"]:
        return f"<p><b>{field_name}:</b> <em>(no span)</em></p>"
    s, e = gf["span"]["char_start"], gf["span"]["char_end"]
    ptype = gf["span"]["provenance_type"]
    color = _COLORS.get(ptype, "#f7ddc5")
    body = (
        text[:s]
        + f'<mark style="background:{color};border-radius:3px;padding:0 2px">'
        + text[s:e] + "</mark>"
        + text[e:]
    ).replace("\n", " ")
    return f"<p><b>{field_name}</b> <code>[{ptype}]</code>: {body}</p>"

display(HTML(
    "<div style='font-family:sans-serif;line-height:1.9;font-size:14px'>"
    + "".join(highlight_html(ABSTRACT, f, result) for f in result.extracted)
    + "</div>"
))

### Two-stage pipeline: pay for LLM inference once

`extract_raw` and `ground` are independently callable. Run the LLM once,
apply different verification strategies to the same output — no re-inference needed.

In [ ]:
# Stage 1: LLM call only → RawExtractionResult
raw = extract_raw(ABSTRACT, SCHEMA, llm, prompt=PROMPT)
print("Raw LLM output:")
for f, v in raw.fields.items():
    print(f"  {f}: {v!r}")

print()

# Stage 2: apply grounding strategies to the same raw output
for mode_name in ("fuzzy", "full", "no-grounding"):
    res = ground(raw, llm, mode=mode_name)
    n_grounded = sum(1 for gf in res.extracted.values() if gf["span"])
    print(f"  mode={mode_name!r:15}  grounded={n_grounded}  quarantined={len(res.quarantined)}")

---
## Part 3 — Optional: few-shot examples and prompt optimization

### Few-shot examples

Pass `examples` to show the model what good extractions look like.
Useful when field boundaries are ambiguous or the schema is non-obvious.

In [ ]:
EXAMPLES = [{
    "text": (
        "A randomised trial of metformin 500mg twice daily enrolled 120 patients "
        "with type 2 diabetes. The primary outcome was HbA1c reduction at 6 months."
    ),
    "fields": {
        "drug":            "metformin 500mg twice daily",
        "sample_size":     "120 patients",
        "primary_outcome": "HbA1c reduction at 6 months",
    },
}]

result_ex = extract(ABSTRACT, SCHEMA, llm, prompt=PROMPT, examples=EXAMPLES, mode="fuzzy")
print("With examples:")
for field, gf in result_ex.extracted.items():
    ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
    print(f"  [{ptype:>12}]  {field}: {gf['value']!r}")

### Prompt optimization

`optimize_prompt` runs iterative refinement: extract → score by grounding rate →
ask the LLM to suggest improvements → repeat. Returns the highest-scoring prompt.

Pass `ground_truth` labels for supervised accuracy scoring.
Benchmarks show **10–20 pp accuracy gains** on clinical NLP tasks after 3–5 iterations.

> **Note:** each iteration calls the LLM twice (extract + refine), so this cell
> takes ~1–2 minutes with `n_iter=2`. In production, run once offline and reuse the result.

In [ ]:
from veritract import optimize_prompt

TRAIN_EXAMPLES = [
    {"text": ABSTRACT},
    {"text": (
        "A placebo-controlled study of rosuvastatin 20mg enrolled 312 patients "
        "with dyslipidaemia. The primary endpoint was LDL-C change from baseline at 24 weeks."
    )},
]

# Unsupervised: score by grounding rate — no labels needed
best_prompt = optimize_prompt(TRAIN_EXAMPLES, SCHEMA, llm, n_iter=2)

# Diff: show what changed vs the original prompt
orig_lines = set(PROMPT.strip().splitlines())
new_lines  = set(best_prompt.strip().splitlines())
added   = new_lines - orig_lines
removed = orig_lines - new_lines
print("Changes from original prompt:")
for line in sorted(removed): print(f"  - {line}")
for line in sorted(added):   print(f"  + {line}")

result_opt = extract(ABSTRACT, SCHEMA, llm, prompt=best_prompt, mode="fuzzy")
print("\nWith optimised prompt:")
for field, gf in result_opt.extracted.items():
    ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
    print(f"  [{ptype:>12}]  {field}: {gf['value']!r}")

---
## Part 4 — The grounding layer

After LLM extraction, every field value is verified against the source text using
two strategies in sequence:

1. **Fuzzy matching** (rapidfuzz `token_set_ratio`) — finds the best-matching
   sentence window in the source. If an exact substring is found, the span is
   pinned to that precise location (`direct`). If only a fuzzy match passes the
   threshold, the window is returned (`paraphrased`).

2. **LLM re-verification** (mode=`"full"` only) — for fields fuzzy matching
   couldn't verify, asks the LLM "does this value appear in the source?".
   If confirmed, the field is promoted to `inferred`.

Values that pass neither are **quarantined** — preserved with a reason for human review.

In [ ]:
# MockLLM simulates a hallucinating model — wrong drug, count, and outcome
hallucinating_llm = MockLLM()
hallucinating_llm.register(ABSTRACT[:30], {
    "drug":            "rosuvastatin 20 mg",   # wrong
    "sample_size":     "250 patients",          # wrong
    "primary_outcome": "cardiovascular events", # wrong
})

raw_h = extract_raw(ABSTRACT, SCHEMA, hallucinating_llm)
print("Raw output — valid JSON, plausible values, but all wrong:")
for field, value in raw_h.fields.items():
    print(f"  {field}: {value!r}")

print()

# Grounding catches every hallucinated value
result_h = ground(raw_h, llm, mode="fuzzy")
print(f"After grounding — grounded: {len(result_h.extracted)}  quarantined: {len(result_h.quarantined)}")
for q in result_h.quarantined:
    print(f"  QUARANTINED  {q['field_name']}: {q['value']!r}")

---
## Part 5 — PDF extraction

`extract_pdf` converts PDF → markdown via docling (handles text, tables, scanned
pages via OCR), chunks the markdown, extracts from each chunk independently,
merges results (preferring compact, high-frequency values across chunks), and
grounds against the full document.

In [ ]:
from veritract import extract_pdf

TECH_SCHEMA = {
    "type": "object",
    "properties": {
        "architecture_type":       {"type": "string"},
        "total_parameters":        {"type": "string"},
        "context_length":          {"type": "string"},
        "pretraining_token_count": {"type": "string"},
        "alignment_and_rl_method": {"type": "string"},
    },
    "required": [
        "architecture_type", "total_parameters", "context_length",
        "pretraining_token_count", "alignment_and_rl_method",
    ],
}

PDF_PROMPT = """\
You are extracting facts from an LLM technical report.
Rules:
- Copy the exact verbatim phrase from the text. Do not paraphrase or use prior knowledge.
- If a field is not present in this specific excerpt, return an empty string.

Fields:
* architecture_type: Model architecture family and variant (dense, MoE, attention type).
* total_parameters: Total parameter count as stated.
* context_length: Maximum context window as stated.
* pretraining_token_count: Total pretraining tokens as stated.
* alignment_and_rl_method: Post-SFT alignment method (RLHF, DPO, GRPO, BOND, etc.)."""

# Download: https://arxiv.org/pdf/2503.19786  (Gemma 3 technical report)
PDF_PATH = "/tmp/llm_reports/gemma3.pdf"

print("Extracting from Gemma 3 technical report...")
result_pdf = extract_pdf(
    PDF_PATH, TECH_SCHEMA, llm,
    chunk_size=8000, chunk_overlap=500,
    mode="fuzzy", prompt=PDF_PROMPT,
)

print("\nGrounded fields:")
for field, gf in result_pdf.extracted.items():
    if gf["value"]:
        ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
        print(f"  [{ptype:>12}]  {field}: {gf['value'][:90]}")

real_q = [q for q in result_pdf.quarantined if q["value"]]
if real_q:
    print("\nQuarantined:")
    for q in real_q:
        print(f"  {q['field_name']}: {q['value'][:60]!r}")

---
## Part 6 — Multimodal: extracting from PDF figures

Pass images alongside text using `images=[b64_string]`.
veritract prepends an image-first preamble automatically so the model attends
to the visual content before reading the extraction instructions.

For figure extraction from PDFs, grounding runs against the **full document text**
so values visible in a chart but discussed in the paper body get verified spans.
Values only readable from the image (chart type, axis ticks) will be quarantined —
an honest signal that those fields came purely from vision.

In [ ]:
import base64, io
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

FIGURE_SCHEMA = {
    "type": "object",
    "properties": {
        "figure_type":     {"type": "string"},
        "models_compared": {"type": "string"},
        "metric":          {"type": "string"},
        "best_result":     {"type": "string"},
        "key_finding":     {"type": "string"},
    },
    "required": ["figure_type", "models_compared", "metric", "best_result", "key_finding"],
}

FIGURE_PROMPT = """\
You are reading a figure from an ML research paper.
Rules:
- Extract only what is directly visible in the figure or stated in the caption.
- Copy exact text labels as they appear. Do not use prior knowledge.
- If a field is not visible or stated, return an empty string.

Fields:
* figure_type: Type of visual — bar chart, line plot, architecture diagram, table, etc.
* models_compared: Model names shown (comma-separated as labelled in the figure).
* metric: What is being measured (exact axis label or caption description).
* best_result: Highest numerical result visible, with model name.
* key_finding: Main takeaway stated in the caption or shown in the figure."""

# Load PDF with image extraction enabled
opts = PdfPipelineOptions()
opts.generate_picture_images = True
opts.images_scale = 2.0
conv = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)})
doc  = conv.convert(PDF_PATH)
full_text = doc.document.export_to_markdown()

# Pick Figure 2 — performance summary chart
qualifying = [
    (p.get_image(doc.document), p.caption_text(doc.document) or "")
    for p in doc.document.pictures
    if p.get_image(doc.document) and p.get_image(doc.document).width >= 200
]
img, caption = qualifying[1]
print(f"Figure: {img.width}×{img.height}px")
print(f"Caption: {caption[:100]}")

buf = io.BytesIO()
img.save(buf, format="PNG")
b64 = base64.b64encode(buf.getvalue()).decode()

# Extract: prompt sees caption + image; grounding searches the full document
raw_fig = extract_raw(caption, FIGURE_SCHEMA, llm,
    prompt=FIGURE_PROMPT, images=[b64], source_type="figure", doc_id="gemma3.pdf")
raw_fig.source_text = full_text   # ground against full doc, not just caption
result_fig = ground(raw_fig, llm=None, mode="fuzzy")

print("\nExtracted:")
for field, gf in result_fig.extracted.items():
    if gf["value"]:
        ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
        print(f"  [{ptype:>12}]  {field}: {gf['value'][:90]}")

if result_fig.quarantined:
    print("\nQuarantined (vision-only — not found in document text):")
    for q in result_fig.quarantined:
        if q["value"]:
            print(f"  {q['field_name']}: {q['value'][:70]!r}")

---
## Benchmark — vs LangExtract on EBM-NLP 2.0

Evaluated on **155 expert-annotated clinical trial abstracts** (fields: `drug`, `sample_size`, `outcome`).
Both systems: `gemma4:e4b` via Ollama, 3 runs × 155 samples, temp=0.3, optimised prompt.

| | veritract | LangExtract |
|---|---|---|
| **Field accuracy** | **87.7% ± 0.3%** | 84.7% ± 1.9% |
| Extraction latency | 28.9s | **5.5s** |
| Verification cost | +5.4s | +0.2s |
| Grounded (has span) | **90.3%** | 85.7% |
| Quarantined | 9.7% | 0%* |
| Missing (never extracted) | **0%** | ~14.3% |
| Quarantine precision | **100%** | — |
| Quarantine recall | 79% | — |

*LangExtract skips fields it can't answer; veritract always emits every schema field.

**Why is veritract slower?** GBNF constrained decoding enforces grammar validity at
every token step, which is inherently slower than LangExtract's QA-span format
(which just copies a verbatim phrase with no grammar checking). The latency tradeoff
buys: guaranteed JSON validity, mandatory emission of all schema fields, and
richer provenance.

---
## Summary

| | veritract | Plain JSON / Instructor |
|---|---|---|
| Extracts structured JSON | ✓ | ✓ |
| Constrained decoding (guaranteed valid JSON) | ✓ GBNF | ✓ (provider-dependent) |
| Every field traced to source span | ✓ | ✗ |
| Hallucinations caught and quarantined | ✓ | ✗ |
| Trust signal per field (not per response) | ✓ | ✗ |
| Fully local / air-gapped | ✓ Ollama | ✗ API required |
| PDF extraction with table + OCR support | ✓ docling | ✗ |
| Multimodal (image + figure) extraction | ✓ | provider-dependent |

**GitHub:** https://github.com/LanGuo/veritract
**Install (from source):** `git clone https://github.com/LanGuo/veritract.git && pip install -e '.[pdf]'`
*(PyPI release coming soon)*